# Reproducible LLM relay experiments on lexigram

This notebook demonstrates the framework's built-in reproducibility path: a **seeded, config-driven** experiment over the Claude relay mapper, with OpenTelemetry tracing (`AITracer`) and structured metrics (`AIMetrics`) recorded on every conversion. No external experiment-tracking service is required — every run is pinned by a digest and persisted under `runs/<run_id>/`.


## The reproducibility contract

1. **Config-driven**: `experiment.yaml` holds every knob (seed, model, iterations, sampling, recording switches).
2. **Seeded**: all synthetic wire payloads, latencies, and token counts come from `random.Random(seed)` — a Python-standard PRNG, stable across runs and platforms.
3. **Digest-pinned**: `sha256(params + metrics + results)` — same seed, same digest.
4. **Evaluator-tracked**: every run is tracked through `lexigram-ai-evaluation` — seed-stable run ids, metric/error streams, and digest-verified per-iteration checkpoints under `runs/<run_id>/`.

In [1]:
from pathlib import Path

from harness import load_config, run_experiment

config = load_config(Path("experiment.yaml"))
config["experiment"]

2026-08-19 21:52:16 [debug    ] module_decorated               _logger_name=lexigram.di.module.decorator controllers=0 exports=0 imports=0 is_global=False module=EvaluationModule providers=0


2026-08-19 21:52:16 [debug    ] module_decorated               _logger_name=lexigram.di.module.decorator controllers=0 exports=0 imports=0 is_global=False module=RelayModule providers=0


{'name': 'llm-relay-probe',
 'description': 'Deterministic conversion probe over the Claude relay mapper',
 'seed': 42,
 'iterations': 5,
 'provider': 'anthropic',
 'model': 'claude-3-5-sonnet',
 'temperature': 0.7,
 'top_p': 0.9,
 'max_tokens': 1024,
 'tracing_enabled': True,
 'metrics_enabled': True}

In [2]:
out = Path("runs")
# Same seed twice, plus a different seed for contrast.
run_a = run_experiment(config, seed=42, out_dir=out)
run_b = run_experiment(config, seed=42, out_dir=out)
run_c = run_experiment(config, seed=7, out_dir=out)
print("run_a:", run_a.run_id)
print("run_b:", run_b.run_id)
print("run_c:", run_c.run_id)

2026-08-19 21:52:16 [info     ] experiment_finished            _logger_name=lexigram.ai.evaluation.tracking finished=ExperimentRun(run_id='llm-relay-probe-42-a1ba4be3', experiment='llm-relay-probe', seed=42, config={'_ablate': 'control', 'description': 'Deterministic conversion probe over the Claude relay mapper', 'iterations': 5, 'max_tokens': 1024, 'metrics_enabled': True, 'model': 'claude-3-5-sonnet', 'name': 'llm-relay-probe', 'provider': 'anthropic', 'seed': 42, 'temperature': 0.7, 'top_p': 0.9, 'tracing_enabled': True}, config_hash='f2c632e4cc4add939eda7e04563a871aee3ab84368b8edaef52422dd7aead470', status=<RunStatus.COMPLETED: 'completed'>, started_at='2026-08-19T13:52:16.930283+00:00', finished_at='2026-08-19T13:52:16.931678+00:00') run_id=llm-relay-probe-42-a1ba4be3 status=completed


2026-08-19 21:52:16 [info     ] experiment_finished            _logger_name=lexigram.ai.evaluation.tracking finished=ExperimentRun(run_id='llm-relay-probe-42-a1ba4be3', experiment='llm-relay-probe', seed=42, config={'_ablate': 'control', 'description': 'Deterministic conversion probe over the Claude relay mapper', 'iterations': 5, 'max_tokens': 1024, 'metrics_enabled': True, 'model': 'claude-3-5-sonnet', 'name': 'llm-relay-probe', 'provider': 'anthropic', 'seed': 42, 'temperature': 0.7, 'top_p': 0.9, 'tracing_enabled': True}, config_hash='f2c632e4cc4add939eda7e04563a871aee3ab84368b8edaef52422dd7aead470', status=<RunStatus.COMPLETED: 'completed'>, started_at='2026-08-19T13:52:16.930283+00:00', finished_at='2026-08-19T13:52:16.936001+00:00') run_id=llm-relay-probe-42-a1ba4be3 status=completed


2026-08-19 21:52:16 [info     ] experiment_finished            _logger_name=lexigram.ai.evaluation.tracking finished=ExperimentRun(run_id='llm-relay-probe-7-8164dff4', experiment='llm-relay-probe', seed=7, config={'_ablate': 'control', 'description': 'Deterministic conversion probe over the Claude relay mapper', 'iterations': 5, 'max_tokens': 1024, 'metrics_enabled': True, 'model': 'claude-3-5-sonnet', 'name': 'llm-relay-probe', 'provider': 'anthropic', 'seed': 42, 'temperature': 0.7, 'top_p': 0.9, 'tracing_enabled': True}, config_hash='f2c632e4cc4add939eda7e04563a871aee3ab84368b8edaef52422dd7aead470', status=<RunStatus.COMPLETED: 'completed'>, started_at='2026-08-19T13:52:16.938743+00:00', finished_at='2026-08-19T13:52:16.939854+00:00') run_id=llm-relay-probe-7-8164dff4 status=completed


run_a: llm-relay-probe-42-a1ba4be3
run_b: llm-relay-probe-42-a1ba4be3
run_c: llm-relay-probe-7-8164dff4


In [3]:
# Both same-seed runs must produce an identical digest...
assert run_a.digest == run_b.digest, "same seed diverged!"
# ...and a different seed must not.
assert run_a.digest != run_c.digest, "different seed collided!"
print("reproducibility: OK — same seed == same digest")
print("digest:", run_a.digest)

reproducibility: OK — same seed == same digest
digest: b1e9e0f75c964ad355dfab278b34106da06cb3850cefab057e506a0e2e70f934


### What gets tracked per run

`AIMetrics` (from `lexigram-ai-observability`) records LLM request counts, token totals, latencies, and cost; `AITracer` emits OpenTelemetry spans. Both are dumped deterministically so a run can be audited or compared later.


In [4]:
print("--- counters ---")
for name, series in run_a.metrics["counters"].items():
    print(f"{name}: {sum(series.values())}")
print("--- histograms ---")
for name, series in run_a.metrics["histograms"].items():
    total = sum(len(v) for v in series.values())
    print(f"{name}: {total} observations")
print("--- totals ---")
print(run_a.result["totals"])

--- counters ---
intelligence_llm_cost_dollars: 0.005406
intelligence_llm_requests_total: 10.0
intelligence_llm_tokens_total: 446.0
--- histograms ---
intelligence_llm_duration_seconds: 5 observations
--- totals ---
{'requests': 5, 'prompt_tokens': 107, 'completion_tokens': 339, 'cost_dollars': 0.005406, 'losses': 0}


### Ablation / error analysis

The harness ships a tiny ablation switch: drop `thinking` blocks from the wire payloads and compare metrics against the control run (`metrics_delta`). Each run also persists a framework `AblationRunner` record — a digest-verified delta between the control and ablated totals checkpoints — and an `ErrorAnalysis` summary (`analysis.json`).

In [5]:
from harness import metrics_delta

ablated = run_experiment(config, seed=42, out_dir=out, ablate="thinking")
deltas = metrics_delta(run_a, ablated)
print("metric deltas when thinking blocks are ablated:")
print(deltas)

2026-08-19 21:52:16 [info     ] experiment_finished            _logger_name=lexigram.ai.evaluation.tracking finished=ExperimentRun(run_id='llm-relay-probe-42-e9f53311', experiment='llm-relay-probe', seed=42, config={'_ablate': 'thinking', 'description': 'Deterministic conversion probe over the Claude relay mapper', 'iterations': 5, 'max_tokens': 1024, 'metrics_enabled': True, 'model': 'claude-3-5-sonnet', 'name': 'llm-relay-probe', 'provider': 'anthropic', 'seed': 42, 'temperature': 0.7, 'top_p': 0.9, 'tracing_enabled': True}, config_hash='824aba78bfbce0812652901ba1c0701d45dd799022e5e0f2535c9653e1d4f740', status=<RunStatus.COMPLETED: 'completed'>, started_at='2026-08-19T13:52:16.954442+00:00', finished_at='2026-08-19T13:52:16.956012+00:00') run_id=llm-relay-probe-42-e9f53311 status=completed


metric deltas when thinking blocks are ablated:
{'intelligence_llm_cost_dollars': -0.000675, 'intelligence_llm_requests_total': 0.0, 'intelligence_llm_tokens_total': -45.0, 'intelligence_llm_duration_seconds': 0}


## Artifacts & replay

Every run is on disk under `runs/<run_id>/`:

- `run.json` — tracking manifest (status, config, seed)
- `metrics.jsonl` — metric stream (name → value per step)
- `analysis.json` — error analysis summary
- `params.json` — pinned config, seed, ablation, config fingerprint
- `metrics.json` — AIMetrics snapshot
- `trace.json` — OTel span list (name + attributes)
- `result.json` — per-iteration conversions + totals
- `checkpoints/iteration_XX.json` — per-step checkpoints
- `checkpoints/baseline.json` — totals checkpoint (control run)
- `checkpoints/ablated-<knob>.json` — totals checkpoint (ablated run)
- `reproducibility.json` — run_id + digest

Replay from the CLI:

```bash
python run_experiment.py --seed 42
python run_experiment.py --seed 42 --ablate thinking
```